<a href="https://colab.research.google.com/github/Saifullah785/machine-learning-engineer-roadmap/blob/main/Lecture_78_Optuna_Basics_yt/Lecture_78_Optuna_Basics_yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Optuna library
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 8.1 MB/s eta 0:00:00


In [ ]:
# Import necessary libraries for model training and hyperparameter tuning
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the Pima Indian Diabetes dataset from a URL
import pandas as pd
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [ ]:
# Load the dataset into a pandas DataFrame and display the head
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
# Handle missing values by replacing zeros with NaN and imputing with the mean
import numpy as np

# Replace zero values with NaN is columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

#check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [ ]:
# Split the data into training and test sets and scale the features
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [ ]:
# Define the objective function for Optuna to optimize RandomForest hyperparameters
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Define the objective function
def objective(trial):
  # Suggest values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators',  50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # create the randomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )
  # Perform 3-fold cross-validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score

In [ ]:
# Create and run an Optuna study to find the best hyperparameters for RandomForest
# create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler()) # we aim to maximize accuracy

# RUn 50 trials to find the best hyperparameters
study.optimize(objective, n_trials=50)

[I 2025-09-12 06:27:57,880] A new study created in memory with name: no-name-865c2f04-c90c-4b59-bf34-34328ddbd700
[I 2025-09-12 06:27:58,682] Trial 0 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 63, 'max_depth': 4}. Best is trial 0 with value: 0.7597765363128491.
[I 2025-09-12 06:28:00,225] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 143, 'max_depth': 15}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-09-12 06:28:01,684] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 130, 'max_depth': 6}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-09-12 06:28:03,218] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 145, 'max_depth': 20}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-09-12 06:28:04,245] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 56, 'max_depth': 20}. Best is trial 1 with value: 0.77094972

In [ ]:
# Print the best trial's accuracy and hyperparameters for RandomForest
# print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 119, 'max_depth': 19}


In [ ]:
# Train and evaluate the RandomForest model with the best hyperparameters
from sklearn.metrics import accuracy_score

#  Train a RandomForestClassifier using the best hyperparameters from optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# print the test acccuracy

print(f'Test accuracy with best hyperparameters: {test_accuracy:.2f}')

Test accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [ ]:
# Import visualization functions from Optuna
# For visualizations

from optuna.visualization import plot_optimization_history, \
plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [ ]:
# Plot the optimization history of the Optuna study
# 1. Optimization History
plot_optimization_history(study).show()

In [ ]:
# Plot the parallel coordinates of the Optuna study
# 2. parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [ ]:
# Plot the slice plot of the Optuna study
# 3.Slice Plot
plot_slice(study).show()

In [ ]:
# Plot the contour plot of the Optuna study
# 4. Contour Plot
plot_contour(study).show()

In [ ]:
# Plot the parameter importances of the Optuna study
# 5. Parameter Importances Plot
plot_param_importances(study).show()

# Optimizing Mulitple ML Models

In [ ]:
# Import additional classifiers for multi-model optimization
# importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [ ]:
# Define the objective function for Optuna to optimize multiple classifiers
# Define the objective function for Optuna
def objective(trial):
  # Choose the algorithm to tune
  classifier_name = trial.suggest_categorical('classifier', ['SVM','RandomForest', 'GradientBoosting'])
  if classifier_name == 'SVM':
    # Suggest hyperparameters for SVM
    c = trial.suggest_float('C', 0.1, 100, log =True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
    model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

  elif classifier_name == 'RandomForest':
    # Suggest hyperparameters for RandomForest
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        random_state=42
    )

  elif classifier_name == 'GradientBoosting':
    # Suggest hyperparameters for GradientBoosting
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rat', 0.01, 0.3, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
  # Perform cross-validation and return the mean accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  return score

In [ ]:
# Create and run an Optuna study to optimize multiple classifiers
# create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-09-12 06:30:45,347] A new study created in memory with name: no-name-7cebbe86-1190-4974-9237-48d0e55ccf4e
[I 2025-09-12 06:30:45,385] Trial 0 finished with value: 0.7597765363128491 and parameters: {'classifier': 'SVM', 'C': 6.646123816110229, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7597765363128491.
[I 2025-09-12 06:30:45,417] Trial 1 finished with value: 0.7672253258845437 and parameters: {'classifier': 'SVM', 'C': 0.7735524248608444, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 1 with value: 0.7672253258845437.
[I 2025-09-12 06:30:46,692] Trial 2 finished with value: 0.7635009310986964 and parameters: {'classifier': 'RandomForest', 'n_estimators': 257, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 1 with value: 0.7672253258845437.
[I 2025-09-12 06:30:46,969] Trial 3 finished with value: 0.7672253258845437 and parameters: {'classifier': 'RandomForest', 'n_estimators': 53, 'max_depth': 16, 'min_

In [ ]:
# Retrieve and print the best trial's accuracy and hyperparameters for multi-model optimization
# Retrieve the best trial
best_trial = study.best_trial

# Print the best trial's accuracy and hyperparameters
print(f'Best trial accuracy: {best_trial.value}')
print(f'Best hyperparameters: {best_trial.params}')

Best trial accuracy: 0.7895716945996275
Best hyperparameters: {'classifier': 'SVM', 'C': 0.1487410667880136, 'kernel': 'linear', 'gamma': 'scale'}


In [ ]:
# Display the dataframe of trials from the Optuna study
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rat,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.759777,2025-09-12 06:30:45.349511,2025-09-12 06:30:45.385290,0 days 00:00:00.035779,6.646124,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.767225,2025-09-12 06:30:45.386368,2025-09-12 06:30:45.417134,0 days 00:00:00.030766,0.773552,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.763501,2025-09-12 06:30:45.418052,2025-09-12 06:30:46.692297,0 days 00:00:01.274245,NaN,False,RandomForest,NaN,NaN,NaN,20.0,2.0,5.0,257.0,COMPLETE
3,3,0.767225,2025-09-12 06:30:46.693663,2025-09-12 06:30:46.969502,0 days 00:00:00.275839,NaN,True,RandomForest,NaN,NaN,NaN,16.0,10.0,9.0,53.0,COMPLETE
4,4,0.741155,2025-09-12 06:30:46.970421,2025-09-12 06:30:51.698354,0 days 00:00:04.727933,NaN,NaN,GradientBoosting,NaN,NaN,0.170633,17.0,2.0,4.0,295.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.783985,2025-09-12 06:31:28.033976,2025-09-12 06:31:28.078552,0 days 00:00:00.044576,0.442434,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.765363,2025-09-12 06:31:28.079714,2025-09-12 06:31:28.146418,0 days 00:00:00.066704,0.249105,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.772812,2025-09-12 06:31:28.147633,2025-09-12 06:31:30.252758,0 days 00:00:02.105125,NaN,False,RandomForest,NaN,NaN,NaN,10.0,8.0,9.0,276.0,COMPLETE
98,98,0.711359,2025-09-12 06:31:30.253695,2025-09-12 06:31:30.301175,0 days 00:00:00.047480,0.124737,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [ ]:
# Count the number of trials for each classifier in the study
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,78
RandomForest,12
GradientBoosting,10


In [ ]:
# Calculate and display the mean accuracy for each classifier in the study
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.745624
RandomForest,0.765673
SVM,0.772334


In [ ]:
# Plot the optimization history of the multi-model Optuna study
# 1.Optimization History
plot_optimization_history(study).show()

In [ ]:
# Plot the slice plot of the multi-model Optuna study
# 3.Slice plot
plot_slice(study).show()

In [ ]:
# Plot the hyperparameter importances of the multi-model Optuna study
# 5. hyperparameter importance
plot_param_importances(study).show()

In [3]:
# Install Optuna library
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 7.7 MB/s eta 0:00:00


In [17]:
# Import necessary libraries for XGBoost optimization
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

In [18]:
# Load and split the Iris dataset
# Load the iris dataset
X,y = load_iris(return_X_y=True)


# split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
# Define the objective function for XGBoost hyperparameter tuning with pruning
# Define the objective function for XGBoost
def objective(trail):
  # Hyperparameter search space
  param = {
      'verbosity': 0,
      'objective': 'multi:softprob',
      'num_class': 3,
      'eval_metric': 'mlogloss', # Ensure that the eval_metric is specified here
      'booster': 'gbtree',
      'lambda': trail.suggest_float('lambda', 1e-8, 1.0, log=True),
      'alpha': trail.suggest_float('alpha', 1e-8, 1.0, log=True),
      'eta': trail.suggest_float('eta', 0.01, 0.3),
      'gamma': trail.suggest_float('gamma', 1e-8, 1.0, log=True),
      'max_depth': trail.suggest_int('max_depth', 3, 9),
      'min_child_weight': trail.suggest_int('min_child_weight', 1, 10),
      'subsample': trail.suggest_float('subsample', 0.4, 1.0),
      'colsample_bytree': trail.suggest_float('colsample_bytree', 0.4, 1.0),
      'n_estimators': 300,
  }
  # Create DMatrix for XGBoost
  dtrain = xgb.DMatrix(X_train, label=y_train)
  dtest = xgb.DMatrix(X_test, label=y_test)

  # Define a pruning callback based on evaluation metrics
  pruning_callback = optuna.integration.XGBoostPruningCallback(trail, "eval-mlogloss") # match the metric name in the evals list

  # Train the model
  bst = xgb.train(
      param,
      dtrain,
      num_boost_round=300,
      evals=[(dtrain,'train'), (dtest, 'eval')],
      early_stopping_rounds=10,
      callbacks=[pruning_callback]

  )
  # predict on the test set
  preds = bst.predict(dtest)
  best_preds = [int(np.argmax(line)) for line in preds]

  # Return accuracy as the objective value
  accuracy = accuracy_score(y_test, best_preds)
  return accuracy

In [20]:
# Create and run an Optuna study with pruning for XGBoost optimization
# create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

[I 2025-09-13 06:38:38,643] A new study created in memory with name: no-name-4eee5bac-1294-43e0-973e-d6de01d33708


[0]	train-mlogloss:0.95053	eval-mlogloss:0.94150
[1]	train-mlogloss:0.83052	eval-mlogloss:0.81590
[2]	train-mlogloss:0.73175	eval-mlogloss:0.71306
[3]	train-mlogloss:0.64655	eval-mlogloss:0.62338
[4]	train-mlogloss:0.57698	eval-mlogloss:0.55095
[5]	train-mlogloss:0.51344	eval-mlogloss:0.48336
[6]	train-mlogloss:0.46104	eval-mlogloss:0.42763
[7]	train-mlogloss:0.41854	eval-mlogloss:0.38368
[8]	train-mlogloss:0.38221	eval-mlogloss:0.34377
[9]	train-mlogloss:0.35061	eval-mlogloss:0.30814
[10]	train-mlogloss:0.32279	eval-mlogloss:0.27652
[11]	train-mlogloss:0.29827	eval-mlogloss:0.24916
[12]	train-mlogloss:0.27726	eval-mlogloss:0.22771
[13]	train-mlogloss:0.25965	eval-mlogloss:0.20878
[14]	train-mlogloss:0.24195	eval-mlogloss:0.18944
[15]	train-mlogloss:0.22824	eval-mlogloss:0.17378
[16]	train-mlogloss:0.21716	eval-mlogloss:0.16207
[17]	train-mlogloss:0.20564	eval-mlogloss:0.14943
[18]	train-mlogloss:0.19735	eval-mlogloss:0.14013
[19]	train-mlogloss:0.18953	eval-mlogloss:0.13148
[20]	train

[I 2025-09-13 06:38:38,848] Trial 0 finished with value: 1.0 and parameters: {'lambda': 1.1471264413212224e-05, 'alpha': 0.6659164267864843, 'eta': 0.12761709167928534, 'gamma': 7.3584050263077905e-06, 'max_depth': 7, 'min_child_weight': 4, 'subsample': 0.5835119912379031, 'colsample_bytree': 0.7711561376974307}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.04006	eval-mlogloss:1.04441
[1]	train-mlogloss:0.91165	eval-mlogloss:0.90619
[2]	train-mlogloss:0.80941	eval-mlogloss:0.79978
[3]	train-mlogloss:0.74081	eval-mlogloss:0.71998
[4]	train-mlogloss:0.68036	eval-mlogloss:0.65854
[5]	train-mlogloss:0.61225	eval-mlogloss:0.58394
[6]	train-mlogloss:0.56676	eval-mlogloss:0.53429
[7]	train-mlogloss:0.53879	eval-mlogloss:0.51404
[8]	train-mlogloss:0.49568	eval-mlogloss:0.46486
[9]	train-mlogloss:0.46171	eval-mlogloss:0.42969
[10]	train-mlogloss:0.42896	eval-mlogloss:0.38882
[11]	train-mlogloss:0.40733	eval-mlogloss:0.36628
[12]	train-mlogloss:0.39813	eval-mlogloss:0.35767
[13]	train-mlogloss:0.37076	eval-mlogloss:0.32757
[14]	train-mlogloss:0.36376	eval-mlogloss:0.31959
[15]	train-mlogloss:0.35714	eval-mlogloss:0.31250
[16]	train-mlogloss:0.34664	eval-mlogloss:0.29824
[17]	train-mlogloss:0.33897	eval-mlogloss:0.28981
[18]	train-mlogloss:0.33335	eval-mlogloss:0.28628
[19]	train-mlogloss:0.33186	eval-mlogloss:0.28440
[20]	train

[I 2025-09-13 06:38:39,055] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.07109138125182143, 'alpha': 0.04064625741667164, 'eta': 0.1164408296462487, 'gamma': 6.0857300507270935e-06, 'max_depth': 8, 'min_child_weight': 7, 'subsample': 0.49773642202748763, 'colsample_bytree': 0.4673146118312575}. Best is trial 0 with value: 1.0.


...

[I 2025-09-13 06:38:44,519] Trial 47 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:0.85303	eval-mlogloss:0.84254
[1]	train-mlogloss:0.61614	eval-mlogloss:0.59438


[I 2025-09-13 06:38:44,543] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:0.88109	eval-mlogloss:0.86239


[I 2025-09-13 06:38:44,594] Trial 49 pruned. Trial was pruned at iteration 1.


In [21]:
# Output the best trial results for XGBoost optimization
# Output the best trial
best_trial = study.best_trial

# Print the best trial's accuracy and hyperparameters
print(f'Best trial accuracy: {best_trial.value}')
print(f'Best hyperparameters: {best_trial.params}')

Best trial accuracy: 1.0
Best hyperparameters: {'lambda': 1.1471264413212224e-05, 'alpha': 0.6659164267864843, 'eta': 0.12761709167928534, 'gamma': 7.3584050263077905e-06, 'max_depth': 7, 'min_child_weight': 4, 'subsample': 0.5835119912379031, 'colsample_bytree': 0.7711561376974307}


In [9]:
# Install optuna-integration with xgboost support
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 3.3 MB/s eta 0:00:00


In [22]:
# Plot intermediate values of the Optuna study for XGBoost optimization
from optuna.visualization import plot_intermediate_values

# 1. plot intermediate values during the trials
plot_intermediate_values(study).show()